# YOLOv5 v5.0 + Google Colab

本 Notebook 包含以下操作：

1. 確認 Colab GPU 執行環境。
2. 掛載 Google Drive。
3. 下載 YOLOv5 `v5.0` 原始碼並安裝套件。
4. 使用示範資料集訓練模型，並透過 TensorBoard 查看訓練紀錄。
5. 使用 `best.pt` 執行物件偵測。
6. 將模型與訓練結果備份到 Google Drive。

開始前，請在 Colab 選擇「執行階段 → 變更執行階段類型」，並選擇可用的 GPU。

參考網址：
- https://blog.csdn.net/qq_51070956/article/details/134503497
- https://github.com/ultralytics/yolov5/tree/v5.0


## 1. 確認執行環境與 GPU

若 `torch.cuda.is_available()` 顯示 `False`，請先在 Colab 設定 GPU 執行階段，再重新執行本格。

In [ ]:
import os
import shutil
import sys
from pathlib import Path

print("Python:", sys.version)
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("檢查 PyTorch 時發生問題：", exc)

!nvidia-smi

## 2. 掛載 Google Drive

訓練結果會先寫入 Colab 本機，訓練完成後再複製到 Drive。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/yolov5_colab')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("備份資料夾：", DRIVE_ROOT)

## 3. 取得 YOLOv5 v5.0 專案並安裝套件

程式固定使用官方 `v5.0` tag。此版本較舊，若目前 Colab 的 Python、PyTorch 或 CUDA 版本不相容，可能仍需依實際錯誤調整環境。

In [ ]:
YOLOV5_REF = "v5.0"
REPO_DIR = Path('/content/yolov5')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 --branch {YOLOV5_REF} https://github.com/ultralytics/yolov5.git /content/yolov5
%cd /content/yolov5
!pip install -r requirements.txt

# 修正部分舊版程式使用、但新版 NumPy 已移除的別名。
import re
for py_file in REPO_DIR.rglob('*.py'):
    original = py_file.read_text(encoding='utf-8')
    patched = original
    for alias, builtin in [('int', 'int'), ('float', 'float'), ('bool', 'bool'), ('object', 'object')]:
        patched = re.sub(rf'\bnp\.{alias}\b', builtin, patched)
    if patched != original:
        py_file.write_text(patched, encoding='utf-8')

# PyTorch 2.6+ 的 torch.load 預設行為可能影響舊版 checkpoint 載入。
# 請只載入來自官方或可信任來源的 .pt 權重檔。
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
print('YOLOv5 version:', YOLOV5_REF)
print('Compatibility patch applied where needed.')

## 4. 設定訓練參數

第一次執行可先用 `coco128.yaml` 和少量 epoch 確認流程。正式訓練時，請依資料集及 GPU 記憶體調整參數。

In [ ]:
DATA_YAML = 'data/coco128.yaml'
WEIGHTS = 'yolov5s.pt'
IMG_SIZE = 640
EPOCHS = 3
BATCH_SIZE = 16
RUN_NAME = 'colab_run'

print({
    'DATA_YAML': DATA_YAML,
    'WEIGHTS': WEIGHTS,
    'IMG_SIZE': IMG_SIZE,
    'EPOCHS': EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'RUN_NAME': RUN_NAME,
})

## 5. 啟動 TensorBoard

先執行本格，再執行下一格開始訓練。訓練開始後，TensorBoard 會顯示產生的紀錄。

In [ ]:
from pathlib import Path
Path('/content/yolov5/runs/train').mkdir(parents=True, exist_ok=True)
%load_ext tensorboard
%tensorboard --logdir /content/yolov5/runs/train

## 6. 開始訓練

第一次執行可能需要下載資料集和預訓練權重。

In [ ]:
%cd /content/yolov5
!python train.py --data {DATA_YAML} --weights {WEIGHTS} --epochs {EPOCHS} --img-size {IMG_SIZE} --batch-size {BATCH_SIZE} --rect --project runs/train --name {RUN_NAME} --exist-ok

## 7. 查看訓練結果

`best.pt` 是驗證表現最佳的權重，`last.pt` 是最後一個 epoch 的權重。

In [ ]:
from IPython.display import display, Image

RUN_DIR = Path('/content/yolov5/runs/train') / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'

print('訓練資料夾存在：', RUN_DIR.exists())
print('best.pt 存在：', BEST_PT.exists())
print('last.pt 存在：', LAST_PT.exists())

for plot_name in ['results.png', 'PR_curve.png', 'confusion_matrix.png']:
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        print(plot_name)
        display(Image(filename=str(plot_path)))

## 8. 使用 `best.pt` 執行物件偵測

預設使用專案內建的 `bus.jpg`。更換自己的圖片時，修改 `SOURCE` 即可。

In [ ]:
if not BEST_PT.exists():
    raise FileNotFoundError(f'找不到權重檔：{BEST_PT}。請先確認訓練已完成。')

SOURCE = 'data/images/bus.jpg'
!python detect.py --weights {BEST_PT} --source {SOURCE} --img-size {IMG_SIZE} --conf-thres 0.25

In [ ]:
from IPython.display import display, Image

detect_dirs = sorted(Path('/content/yolov5/runs/detect').glob('exp*'), key=lambda p: p.stat().st_mtime)
if detect_dirs:
    newest_detect_dir = detect_dirs[-1]
    images = [p for p in newest_detect_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
    print('偵測結果資料夾：', newest_detect_dir)
    if images:
        display(Image(filename=str(images[0])))
    else:
        print('資料夾中沒有找到輸出的圖片檔。')
else:
    print('尚未找到 runs/detect 的輸出，請確認偵測指令執行成功。')

## 9. 備份模型與訓練紀錄到 Google Drive

請確認本格成功執行後，再關閉 Colab 執行階段。

In [ ]:
DRIVE_RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
DRIVE_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)

if RUN_DIR.exists():
    shutil.copytree(RUN_DIR, DRIVE_RUN_DIR, dirs_exist_ok=True)
    print('訓練輸出已備份到：', DRIVE_RUN_DIR)
else:
    print('找不到訓練輸出資料夾，請先完成訓練。')

if BEST_PT.exists():
    print('best.pt：', BEST_PT)
    print('Drive 備份：', DRIVE_RUN_DIR / 'weights' / 'best.pt')

## 10. 套用到自己的資料集

資料夾結構範例：

```text
my_dataset/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/
└── data.yaml
```

`data.yaml` 範例：

```yaml
path: /content/my_dataset
train: images/train
val: images/val
names:
  0: person
  1: car
```

每張圖片需有檔名對應的 YOLO 格式 `.txt` 標註檔，每行格式為 `class_id x_center y_center width height`，座標與寬高需正規化到 0–1。將資料放到 Colab 後，把 `DATA_YAML` 改成資料集設定檔的路徑，例如：

```python
DATA_YAML = '/content/my_dataset/data.yaml'
EPOCHS = 50
```

若要使用完整 COCO 資料集，可將 `DATA_YAML` 改成 `data/coco.yaml`；該資料集需要更多下載時間、儲存空間與 GPU 資源。`EPOCHS = 3` 僅供測試流程使用。

### 常用命令

```bash
!nvidia-smi
%pwd
%cd /content/yolov5
!python train.py --help
!python detect.py --help
```

此為復刻版本。